### Notebook for creating colonies data

Data Sources:
<br>
colonies: https://ourworldindata.org/grapher/european-overseas-colonies-and-their-colonizers?time=1803 (last accessed 13.09.2025)
<br>
countries: https://www.naturalearthdata.com/downloads/50m-cultural-vectors/50m-admin-0-countries-2/ (last accessed 13.09.2025)

This notebook creates yearly shapefiles on countries and if they belong to a colonising country (period 1462-2022)

In [ ]:
#importing relevant packages
import pandas as pd
import os
import geopandas as gpd

In [2]:
#data directory for colonies data
data_dir = "../data/colonies/"
basemap_dir = "../data/basemap/"

In [3]:
#fetch the data from ourworldindata
colonies_original = pd.read_csv("https://ourworldindata.org/grapher/european-overseas-colonies-and-their-colonizers.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

In [4]:
colonies_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88747 entries, 0 to 88746
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Entity             88747 non-null  object
 1   Code               88747 non-null  object
 2   Year               88747 non-null  int64 
 3   colonizer_grouped  88747 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.7+ MB


In [5]:
colonies_original.head()

,Entity,Code,Year,colonizer_grouped
0,Afghanistan,AFG,1462,zzz. Not colonized
1,Afghanistan,AFG,1463,zzz. Not colonized
2,Afghanistan,AFG,1464,zzz. Not colonized
3,Afghanistan,AFG,1465,zzz. Not colonized
4,Afghanistan,AFG,1466,zzz. Not colonized


In [6]:
#export each year's data as a separate CSV file to the specified folder
output_folder = data_dir+"yearly"

#check if directory exists, if not create it
os.makedirs(output_folder, exist_ok=True)

#loop through each year and save the corresponding data
for year, data in colonies_original.groupby('Year'):
    data.to_csv(os.path.join(output_folder, f'colonies_{year}.csv'), index=False)

In [7]:
#path to the shapefile from Natural Earth Data
#for joining the colonies data with geodata
shapefile_path = basemap_dir+"ne_50m_admin_0_countries/ne_50m_admin_0_countries.shp"

#read the shapefile
gdf_countries = gpd.read_file(shapefile_path)

#preview columns to find the country code column ISO_A3
print(gdf_countries[['ADMIN', 'ISO_A3']].head())

       ADMIN ISO_A3
0   Zimbabwe    ZWE
1     Zambia    ZMB
2      Yemen    YEM
3    Vietnam    VNM
4  Venezuela    VEN


In [8]:
#paths
shapefile_path = basemap_dir+"/ne_50m_admin_0_countries/ne_50m_admin_0_countries.shp"
yearly_folder = data_dir+"yearly"
output_folder = data_dir+"yearly_shapefiles"

#make sure output directory exists
os.makedirs(output_folder, exist_ok=True)

#read shapefile
gdf_countries = gpd.read_file(shapefile_path)

#reproject to EPSG:4326 (WGS 84)
gdf_countries.to_crs(epsg=4326, inplace=True)

#loop through each yearly CSV and join with shapefile, then export as shapefile
for file in os.listdir(yearly_folder):
    if file.endswith('.csv'):
        year = file.split('_')[-1].replace('.csv', '')
        colonies_original_year = pd.read_csv(os.path.join(yearly_folder, file))

        #limit to relevant columns
        colonies_original_year = colonies_original_year[['Entity', 'Code', 'Year', 'colonizer_grouped']]

        #join datasets on country code columns (ISO_A3 and Code)
        merged = gdf_countries.merge(colonies_original_year, left_on='ISO_A3', right_on='Code', how='left')

        #limit to relevant columns
        merged = merged[['ADMIN', 'ISO_A3', 'Entity', 'Code', 'Year', 'colonizer_grouped', 'geometry']]

        # Export to yearly GeoPackage-files
        merged.to_file(os.path.join(output_folder, f"colonies_{year}.gpkg"), driver="GPKG")

print("Yearly shapefiles created successfully.")

Yearly shapefiles created successfully.
